# Creating Star Schema 

This notebook explores the creation of a Star Schema from raw data using Apache Spark.

The data used in this notebook is from the Brazilian Basic Education Census, which is available at the [Brazilian government's open data portal](https://download.inep.gov.br/dados_abertos)

## Creating SparkSession

The first step is to create a SparkSession.

In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

import psycopg2
from datetime import datetime

from itertools import chain

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("ETL-Censo") \
    .config("spark.jars", "/usr/local/spark/jars/postgresql-42.6.0.jar") \
    .getOrCreate()

## Transforming CSV to Parquet

This transformation aims to increase the speed of data loading by using Parquet files.

In [3]:
# Read CSV data
data_csv = (
    spark
    .read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .option("encoding", "latin1")
    .load("/home/jovyan/data/*.csv")  # ← absolute path
)

Write to Parquet file

In [4]:
data_csv.write.parquet("./data/censo_escolar.parquet")

AnalysisException: path file:/home/jovyan/work/data/censo_escolar.parquet already exists.

Reading from Parquet file

In [3]:
data = (
    spark
    .read
    .format("parquet")
    .load("./data/censo_escolar.parquet/")
)

## Dimensions

The dimensions are...

---

The code below creates the dimensions based on a configuration dict.

```json
{
    "DIM_NAME":{
        # The fields are the table columns
        "fields":[
            {
                "field":"FIELD_1_NAME", # The column name
                "type":"FIELD_1_TYPE",  # The column type in spark
            },
            {
                "field":"FIELD_2_NAME",
                "type":"FIELD_2_TYPE",
            },
            ...
        ],
    }
}
```

In [4]:
INTEGER_DIMENSIONS = [
    "TP_DEPENDENCIA",           # The school administration (state, city, private) 
    "TP_LOCALIZACAO",           # The school location (urban rural)
    "IN_AGUA_POTAVEL",          # has access to drinkable water 
    "IN_ENERGIA_INEXISTENTE",   # has (NOT) access to energy
    "IN_ESGOTO_INEXISTENTE",    # has (NOT) access to energy
    "IN_BANHEIRO",              # has restroom
    "IN_BIBLIOTECA",            # has library
    "IN_REFEITORIO",            # has canteen 
    "IN_COMPUTADOR",            # has computer
    "IN_INTERNET",              # has internet
    "IN_EQUIP_NENHUM"           # no electronic equipment
]

DIMENSION_TABLES_CONFIG = {
    "DIM_LOCAL":{
        "fields": [
            {"field":"NO_UF", "type":"string",},        # State's name 
            {"field":"SG_UF", "type":"string",},        # State's abbreviation
            {"field":"CO_UF", "type":"string",},        # State's code
            {"field":"NO_MUNICIPIO", "type":"string",}, # City's name
            {"field":"CO_MUNICIPIO", "type":"string",}  # City's code
        ]
    },
}

DIMENSION_TABLES_CONFIG.update(
    {
        "DIM_"+dimension.upper():{
            "fields": [
                {"field":dimension, "type":"integer"} 
            ]
        }
        for dimension in INTEGER_DIMENSIONS
    }
)

### Creating dimensions table in Postgres
-----------------------------------------------------------------------------------------------------------------------

Defining the properties of the Postgres Connection

In [5]:
POSTGRES_USER = "censo"
POSTGRES_PASSWORD = "123"
POSTGRES_DB = "censo_escolar"

# Used to connect to the PostgreSQL database server
# in spark session
POSTGRES_CONFIG = {
    "url":f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
    "properties":{
        "user":POSTGRES_USER, 
        "password":POSTGRES_PASSWORD,
        "driver":"org.postgresql.Driver",
    },
}

Establishing connection to Postgres

In [6]:
conn = psycopg2.connect(
    host="postgres",
    port="5432",

    dbname=POSTGRES_DB,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD
)

Function to create a Dimension table in Postgres using the configuration in DIMENSION_TABLES_CONFIG and adding an id column

The code below creates the dimensions
Spark will create a table with the name of the dimension and the columns in the configuration

In [7]:
# Write data to Postgres
# Using the configuration in DIMENSION_TABLES_CONFIG
# With id as the primary key

for table_name, table_config in DIMENSION_TABLES_CONFIG.items():
    
    print(f"[{datetime.now()}] Writing {table_name}")
    
    data\
    .select(
        [
            F
            .col(field["field"])
            .cast(field["type"])
            .alias(field["field"])
            
            for field
            in table_config["fields"]
        ]
    )\
    .distinct()\
    .withColumn(
        "id", F.monotonically_increasing_id()
    )\
    .write\
    .jdbc(
        **POSTGRES_CONFIG,
        table=table_name,
        mode="overwrite"
    )
    
    print(f"[{datetime.now()}] Wrote {table_name}")
    # Define id as the primary key
    cursor = conn.cursor()
    cursor.execute(
        f"ALTER TABLE {table_name} ADD PRIMARY KEY (id);"
    )
    cursor.close()
    conn.commit()

    print(f"[{datetime.now()}] Added primary key to {table_name}")
    print(f"[{datetime.now()}] Done")

[2026-05-12 18:20:41.705910] Writing DIM_LOCAL
[2026-05-12 18:20:45.079734] Wrote DIM_LOCAL
[2026-05-12 18:20:45.120862] Added primary key to DIM_LOCAL
[2026-05-12 18:20:45.120910] Done
[2026-05-12 18:20:45.120915] Writing DIM_TP_DEPENDENCIA
[2026-05-12 18:20:46.034466] Wrote DIM_TP_DEPENDENCIA
[2026-05-12 18:20:46.043076] Added primary key to DIM_TP_DEPENDENCIA
[2026-05-12 18:20:46.043114] Done
[2026-05-12 18:20:46.043121] Writing DIM_TP_LOCALIZACAO
[2026-05-12 18:20:46.751530] Wrote DIM_TP_LOCALIZACAO
[2026-05-12 18:20:46.761475] Added primary key to DIM_TP_LOCALIZACAO
[2026-05-12 18:20:46.761528] Done
[2026-05-12 18:20:46.761535] Writing DIM_IN_AGUA_POTAVEL
[2026-05-12 18:20:47.425660] Wrote DIM_IN_AGUA_POTAVEL
[2026-05-12 18:20:47.434619] Added primary key to DIM_IN_AGUA_POTAVEL
[2026-05-12 18:20:47.434657] Done
[2026-05-12 18:20:47.434666] Writing DIM_IN_ENERGIA_INEXISTENTE
[2026-05-12 18:20:48.078224] Wrote DIM_IN_ENERGIA_INEXISTENTE
[2026-05-12 18:20:48.087384] Added primary key

## Facts table

The definition of the facts table follows a different pattern than the dimensions.

The table schema is previously defined to properly define the foreing keys.


Defining the facts table schema
Metrics + Facts + Dimensions (Foreign Keys)

In [8]:
FACT_TABLE_NAME = "FACT_CENSO_ESCOLAR"

FACT_COLUMNS = [
    "QT_DOC_BAS",  	# Number of Teachers in the basic education (TOTAL)
    "QT_DOC_INF",	  # Number of Teachers in the basic education (child education)
    "QT_DOC_FUND",	# Number of Teachers in the basic education (elementary education)
    "QT_DOC_MED",	  # Number of Teachers in the basic education (high school)
  
    "QT_MAT_BAS",	  # Number of enrollments in the basic education (TOTAL)
    "QT_MAT_INF",	  # Number of enrollments in the basic education (child education)
    "QT_MAT_FUND",	# Number of enrollments in the basic education (elementary education)
    "QT_MAT_MED",	  # Number of enrollments in the basic education (high school)

    "QT_MAT_BAS_ND",	      # Number of enrollments in the basic education - Skin color/Race Not Declared
    "QT_MAT_BAS_BRANCA",	  # Number of enrollments in the basic education - Skin color/Race Branco
    "QT_MAT_BAS_PRETA",	    # Number of enrollments in the basic education - Skin color/Race Preto
    "QT_MAT_BAS_PARDA",	    # Number of enrollments in the basic education - Skin color/Race Parda
    "QT_MAT_BAS_AMARELA",	  # Number of enrollments in the basic education - Skin color/Race Amarela
    "QT_MAT_BAS_INDIGENA",	# Number of enrollments in the basic education - Skin color/Race Indígena
    
    "NU_ANO_CENSO"          # Census' year
]

FACT_CONFIG = {
    fact:{
        "fields": [
            {"field":fact, "type":"integer"}
        ]
    }
    for fact in FACT_COLUMNS
}

DIMENSION_ID_CONFIG = {
    table_name:[
        field['field'] 
        for field 
        in table_fields['fields']
    ]
    for table_name, table_fields in DIMENSION_TABLES_CONFIG.items()
}

FACT_TABLE_ALL_COLUMNS_ORDERED = FACT_COLUMNS + list(map(lambda col:"ID_"+col, DIMENSION_ID_CONFIG.keys()))

Before inserting the data into the facts table, we need to create a function to create the facts table in Postgres

The code below creates the facts table in Postgres using the configuration in FACT_TABLES_CONFIG and adding an id column for each dimension

In [9]:
# Create fact table
# Using the configuration in FACT_CONFIG
# With id as the primary key

# Avoid inserting a backslash into a f-string
comma_break_line = ",\n\t\t\t"
facts_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {FACT_TABLE_NAME} (
        id SERIAL PRIMARY KEY,
        { 
            comma_break_line.join(
                [
                    f"{field} INTEGER" 
                    for field in FACT_COLUMNS
                ]
                +[
                    f"ID_{dim_table} BIGINT"
                    for dim_table in DIMENSION_ID_CONFIG.keys()
                ]
            )
        }
    );
    
    -- Adding Foreign Keys
    ALTER TABLE {FACT_TABLE_NAME}
    {
        comma_break_line.join(
            [
                f"ADD CONSTRAINT {FACT_TABLE_NAME}_{dim_table}_fk FOREIGN KEY (ID_{dim_table}) REFERENCES {dim_table}(id)"
                for dim_table in DIMENSION_ID_CONFIG.keys()
            ]
        )
    }
"""

In [10]:
print(facts_table_sql)


    CREATE TABLE IF NOT EXISTS FACT_CENSO_ESCOLAR (
        id SERIAL PRIMARY KEY,
        QT_DOC_BAS INTEGER,
			QT_DOC_INF INTEGER,
			QT_DOC_FUND INTEGER,
			QT_DOC_MED INTEGER,
			QT_MAT_BAS INTEGER,
			QT_MAT_INF INTEGER,
			QT_MAT_FUND INTEGER,
			QT_MAT_MED INTEGER,
			QT_MAT_BAS_ND INTEGER,
			QT_MAT_BAS_BRANCA INTEGER,
			QT_MAT_BAS_PRETA INTEGER,
			QT_MAT_BAS_PARDA INTEGER,
			QT_MAT_BAS_AMARELA INTEGER,
			QT_MAT_BAS_INDIGENA INTEGER,
			NU_ANO_CENSO INTEGER,
			ID_DIM_LOCAL BIGINT,
			ID_DIM_TP_DEPENDENCIA BIGINT,
			ID_DIM_TP_LOCALIZACAO BIGINT,
			ID_DIM_IN_AGUA_POTAVEL BIGINT,
			ID_DIM_IN_ENERGIA_INEXISTENTE BIGINT,
			ID_DIM_IN_ESGOTO_INEXISTENTE BIGINT,
			ID_DIM_IN_BANHEIRO BIGINT,
			ID_DIM_IN_BIBLIOTECA BIGINT,
			ID_DIM_IN_REFEITORIO BIGINT,
			ID_DIM_IN_COMPUTADOR BIGINT,
			ID_DIM_IN_INTERNET BIGINT,
			ID_DIM_IN_EQUIP_NENHUM BIGINT
    );
    
    -- Adding Foreign Keys
    ALTER TABLE FACT_CENSO_ESCOLAR
    ADD CONSTRAINT FACT_CENSO_ESCOLAR_DIM_LOCAL_fk FORE

Executing the function to create the facts table in Postgres

In [11]:
print(f"[{datetime.now()}] Creating facts table")

cursor = conn.cursor()
try:
    cursor.execute(facts_table_sql)
    cursor.close()
    conn.commit()
except Exception as e:
    print(e)
    conn.rollback()
    cursor.close()
else:
    print(f"[{datetime.now()}] Created facts table")
    print(f"[{datetime.now()}] Done")


[2026-05-12 18:21:05.950820] Creating facts table
[2026-05-12 18:21:06.023342] Created facts table
[2026-05-12 18:21:06.023382] Done


Extracting the data

In [12]:
facts_data = data\
    .select(
        [
            *chain(
                *DIMENSION_ID_CONFIG.values(), 
                FACT_CONFIG.keys()
            )
        ]
    )

Adding the ids for each dimension

In [13]:
# Joining the id of the dimensions

for table_name, table_fields in DIMENSION_ID_CONFIG.items():
    
    # Read the dimension data from Postgres
    dim_table = spark.read\
        .jdbc(
            **POSTGRES_CONFIG,
            table=table_name,
        )\
        .withColumnRenamed("id", f"ID_{table_name}")
    
    # Join the dimension data with the fact data
    facts_data = facts_data\
        .join(
            dim_table,
            on=table_fields,
            how="left"
        )\
        .drop(*table_fields)

Saving data to Postgres

In [ ]:
facts_data\
    .select(*FACT_TABLE_ALL_COLUMNS_ORDERED)\
    .repartition(10)\
    .write\
    .option("batchsize", 10000)\
    .option("numPartitions", 10)\
    .jdbc(
        **POSTGRES_CONFIG,
        table=FACT_TABLE_NAME,
        mode="append"
    )

## References

https://towardsdatascience.com/explaining-technical-stuff-in-a-non-techincal-way-apache-spark-274d6c9f70e9

https://towardsdatascience.com/adding-sequential-ids-to-a-spark-dataframe-fa0df5566ff6

https://sparkbyexamples.com/pyspark/pyspark-read-and-write-parquet-file/

https://www.psycopg.org/docs/usage.html
